In [ ]:
import os
import sys

import numpy as np
import torch
import torch.nn as nn

from lib.dataset.const import INITIAL_JOINT_ANGLE
from lib.models.backbones.Resnet import get_resnet
from lib.models.backbones.HRnet import get_hrnet
from lib.utils.utils import set_random_seed, create_logger, get_dataloaders, get_scheduler, resume_run, save_checkpoint
from lib.utils.urdf_robot import URDFRobot
from easydict import EasyDict

torch.cuda.set_device(0)
args_for_data = EasyDict({
    "urdf_robot_name": "panda",
    "train_ds_names": "./data/dream/real/panda-orb",
    "val_ds_names": None,
    "image_size": 256.0,

    "jitter": True,
    "other_aug": True,
    "occlusion": True,
    "occlu_p": 0.5,
    "padding": False,
    "fix_truncation": False,
    "truncation_padding": [120, 120, 120, 120],
    "rootnet_flip": False,


    "backbone_name": "resnet50",
    "rootnet_backbone_name": "hrnet32",
    "rootnet_image_size": 256.0,
    "other_image_size": 256.0,
    "use_rpmg": False,
    "batch_size": 48,
    "epoch_size": 104950,
    "n_epochs": 700,
    "n_dataloader_workers": 6,
    "save_epoch_interval": None,
    "clip_gradient": 5.0})

robot = URDFRobot(args_for_data.urdf_robot_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds_iter_train, test_loader_dict = get_dataloaders(args_for_data)

init_param_dict = {
        "robot_type" : args_for_data.urdf_robot_name,
        "pose_params": INITIAL_JOINT_ANGLE,
        "cam_params": np.eye(4,dtype=float),
        "init_pose_from_mean": True
    }

/home/ruihengwang/robot_pose_estim/Holistic-Robot-Pose-Estimation/lib/utils/urdfpytorch/urdf.py:2169: RuntimeWarning: invalid value encountered in divide
  value = value / np.linalg.norm(value)
104972it [00:00, 173086.30it/s]
5997it [00:00, 182252.69it/s]
5997it [00:00, 185021.05it/s]
6394it [00:00, 187040.16it/s]
4966it [00:00, 190034.43it/s]
5944it [00:00, 186866.22it/s]
32315it [00:00, 174507.51it/s]

len(ds_iter_train):  2187
len(ds_iter_test_dr):  125
len(ds_iter_test_photo):  125
len(ds_iter_test_azure):  134
len(ds_iter_test_kinect):  104
len(ds_iter_test_realsense):  124
len(ds_iter_test_orb):  674


In [ ]:
model_backbone = get_resnet(args_for_data.backbone_name)
depth_backbone = get_hrnet(type_name=32, num_joints=7, depth_dim=64,
                            pretrain=True, generate_feat=True, generate_hm=False)
# model_backbone
from tqdm import tqdm
from torchnet.meter import AverageValueMeter
from lib.utils.utils import cast
from lib.utils.integral import HeatmapIntegralJoint, HeatmapIntegralPose
from lib.models.class_head import ClassificationHead
from lib.models.tokenizer import VectorQuantizeTokenizer
args_for_tokenizer = EasyDict({
    "encoder_num_blocks": 3,
    "num_joints": 7,
    "encoder_num_blocks": 4,
    "encoder_token_inter_dim": 64,
    "encoder_hidden_dim": 128,
    "encoder_hidden_inter_dim": 32,
    "encoder_dropout": 0.1,
    "token_num": 64,
    "token_class_num": 1024,
    "token_dim": 128,
    "ema_decay": 0.99,
    "decoder_num_blocks": 4,
    "decoder_hidden_dim": 128,
    "decoder_hidden_inter_dim": 32,
    "decoder_token_inter_dim": 64,
    "decoder_p_dropout": 0.1,
    "tokenizer_pretrained": "./experiments/panda_full_w_vq_3d_synth_dr/tokenizer_pretrained/epoch_50_tokenizer.pk"
})
integral_layer = HeatmapIntegralPose(backbone=args_for_data.backbone_name,
                                     num_joints=7,
                                     depth_dim=64,
                                     height_dim=64,
                                     width_dim=64,
                                     norm_type="softmax",
                                     image_size=256.0,
                                     bbox_3d_shape=[1300, 1300, 1300],
                                     root_id=3,
                                     fixroot=True)

tokenizer = VectorQuantizeTokenizer(data_dim=3,
                                    encoder_num_blocks=args_for_tokenizer.encoder_num_blocks,
                                    num_joints=args_for_tokenizer.num_joints,
                                    encoder_token_inter_dim=args_for_tokenizer.encoder_token_inter_dim,
                                    encoder_hidden_dim=args_for_tokenizer.encoder_hidden_dim,
                                    encoder_hidden_inter_dim=args_for_tokenizer.encoder_hidden_inter_dim,
                                    encoder_dropout=args_for_tokenizer.encoder_dropout,
                                    token_num=args_for_tokenizer.token_num,
                                    token_class_num=args_for_tokenizer.token_class_num,
                                    token_dim=args_for_tokenizer.token_dim,
                                    ema_decay=args_for_tokenizer.ema_decay,
                                    decoder_num_blocks=args_for_tokenizer.decoder_num_blocks,
                                    decoder_hidden_dim=args_for_tokenizer.decoder_hidden_dim,
                                    decoder_hidden_inter_dim=args_for_tokenizer.decoder_hidden_inter_dim,
                                    decoder_token_inter_dim=args_for_tokenizer.decoder_token_inter_dim,
                                    decoder_p_dropout=args_for_tokenizer.decoder_p_dropout,
                                    stage="classifier")
tokenizer.init_weights(pretrained=args_for_tokenizer.tokenizer_pretrained)




/home/ruihengwang/miniconda3/envs/hoilistic/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ruihengwang/miniconda3/envs/hoilistic/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Initialized resnet50 from model zoo
Loading hrnet pretrained weights (ImageNet) from ./models/hrnet_w32-36af842e_roc.pth
Loading pretrained tokenizer....
Tokenizer pretrain weight loaded successfully!


In [4]:
args_for_class_head = EasyDict(
{
    "class_in_channels": 2048,
    "class_hidden_dim": 128,
    "class_num_blocks": 3,
    "class_hidden_inter_dim": 64,
    "class_token_inter_dim": 64,
    "class_conv_channels": 128,
    "class_p_dropout": 0.1
})
class_head = ClassificationHead(
    in_channels=args_for_class_head.class_in_channels, 
    image_size=(256.0, 256.0),
    num_joints=args_for_tokenizer.num_joints,
    conv_channels=args_for_class_head.class_conv_channels,
    hidden_dim=args_for_class_head.class_hidden_dim,
    num_blocks=args_for_class_head.class_num_blocks,
    hidden_inter_dim=args_for_class_head.class_hidden_inter_dim,
    token_inter_dim=args_for_class_head.class_token_inter_dim,
    dropout=args_for_class_head.class_p_dropout,
    token_num=args_for_tokenizer.token_num,
    token_class_num=args_for_tokenizer.token_class_num,
    tokenizer=tokenizer
)


In [5]:
torch.cuda.set_device(4)
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
from torchnet.meter import AverageValueMeter
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


log_dir = "new_exp/logdir"
os.makedirs(log_dir, exist_ok=True)

tb_writer = SummaryWriter(log_dir=os.path.join(log_dir, "tensorboard"))
log_csv = os.path.join(log_dir, "training_loss_vq.csv")
columns = ["epoch", "step", "loss_total", "loss_3dkp", "loss_token"]
df = pd.DataFrame(columns=columns)
losses = []
class myModelIntegral(nn.Module):

    def __init__(self, model_backbone, depth_backbone, integral_layer=None):
        super().__init__()
        self.model_backbone = model_backbone
        self.depth_backbone = depth_backbone
        self.inplanes = 2048
        self.integral_layer = integral_layer
        self.kps_need_depth = [3]
        self.deconv_dim = [256,256,256]
        self.depth_layer = nn.Conv2d(
            in_channels=self.inplanes,
            out_channels=1, 
            kernel_size=1,
            stride=1,
            padding=0
        )
        self.deconv_layers = self._make_deconv_layer()
        self.avgpool = nn.AvgPool2d(int(256/32), stride=1)
        self.final_layer = nn.Conv2d(self.deconv_dim[2], 7 * 64, kernel_size=1, stride=1, padding=0)

    def _make_deconv_layer(self):
        deconv_layers = []
        deconv1 = nn.ConvTranspose2d(
            2048, self.deconv_dim[0], kernel_size=4, stride=2, padding=int(4 / 2) - 1, bias=False)
        bn1 = nn.BatchNorm2d(self.deconv_dim[0])
        deconv2 = nn.ConvTranspose2d(
            self.deconv_dim[0], self.deconv_dim[1], kernel_size=4, stride=2, padding=int(4 / 2) - 1, bias=False)
        bn2 = nn.BatchNorm2d(self.deconv_dim[1])
        deconv3 = nn.ConvTranspose2d(
            self.deconv_dim[1], self.deconv_dim[2], kernel_size=4, stride=2, padding=int(4 / 2) - 1, bias=False)
        bn3 = nn.BatchNorm2d(self.deconv_dim[2])

        deconv_layers.append(deconv1)
        deconv_layers.append(bn1)
        deconv_layers.append(nn.ReLU(inplace=True))
        deconv_layers.append(deconv2)
        deconv_layers.append(bn2)
        deconv_layers.append(nn.ReLU(inplace=True))
        deconv_layers.append(deconv3)
        deconv_layers.append(bn3)
        deconv_layers.append(nn.ReLU(inplace=True))

        return nn.Sequential(*deconv_layers)

    def forward(self,  x_reg_input, x_root_input, **kwargs):
        batch_size = x_reg_input.shape[0]
        x_reg_input = x_reg_input.to(torch.float)
        x_root_input = x_root_input.to(torch.float)

        k_value = kwargs.get("k_value", None)
        
        K = kwargs.get("K", None)

        img_feat = self.depth_backbone(x_root_input)
        img_feat = torch.unsqueeze(img_feat,2)
        img_feat = torch.unsqueeze(img_feat,3)
        gamma = self.depth_layer(img_feat)
        gamma = gamma.view(-1,1)
        pred_depth = gamma * k_value.view(-1,1)
        pred_depth = pred_depth.reshape(img_feat.size(0), 1) / 1000.0
        pred_depth = cast(pred_depth, device)

        root_trans_from_rootnet = torch.zeros((batch_size, 3)).float()
        root_trans_from_rootnet[:,2:3] = pred_depth

        x_out = self.model_backbone(x_reg_input)
        xf = self.avgpool(x_out)
        out = self.deconv_layers(x_out)
        out = self.final_layer(out)
        pred_uvd, pred_xyz_int = self.integral_layer(out, root_trans=root_trans_from_rootnet, K=K)
        return pred_uvd, pred_xyz_int
    
model = myModelIntegral(model_backbone, depth_backbone, integral_layer)

args_for_optim = EasyDict({
    "lr": 1e-4,
    "weight_decay": 0.0,
    "use_schedule": True,
    "schedule_type": "exponential",
    "n_epochs_warmup": 0,
    "start_decay": 45,
    "end_decay": 100,
    "final_decay": 0.01,
    "exponent": 0.95,
    "n_epochs": 20,
    "batch_size": 48,
    "clip_gradient": 5.0
})
optimizer = torch.optim.Adam(model.parameters(), lr=args_for_optim.lr, weight_decay=args_for_optim.weight_decay)
curr_max_auc = 0.0
curr_max_auc_4real = { "azure": 0.0, "kinect": 0.0, "realsense": 0.0, "orb": 0.0 }
start_epoch, last_epoch, end_epoch = 0, -1, args_for_optim.n_epochs
lr_scheduler = get_scheduler(args_for_optim, optimizer, last_epoch)

for epoch in range(start_epoch, end_epoch + 1):
    model.train()
    model.to(device)
    print("Epoch: {}".format(epoch))
    iterator = tqdm(ds_iter_train, dynamic_ncols=True)
    losses_3dkp = AverageValueMeter()  # 使用 torchnet 的 AverageValueMeter

    for batch_idx, input_batch in enumerate(iterator):
        optimizer.zero_grad()
        root_images = cast(input_batch["root"]["images"], device).float() / 255.
        root_K = cast(input_batch["root"]["K"], device).float()
        reg_images = cast(input_batch["other"]["images"], device).float() / 255.
        other_K = cast(input_batch["other"]["K"], device).float()
        bboxes = cast(input_batch["root"]["bbox_gt2d_extended"], device).float()

        batch_size = reg_images.shape[0]
        gt_keypoints3d = cast(input_batch["other"]["keypoints_3d"], device).float()
        gt_keypoints2d = cast(input_batch["other"]["keypoints_2d"], device).float()

        real_bbox = torch.tensor([1000.0, 1000.0]).to(torch.float32)
        fx, fy = root_K[:, 0, 0], root_K[:, 1, 1]
        area = torch.max(torch.abs(bboxes[:, 2] - bboxes[:, 0]), torch.abs(bboxes[:, 3] - bboxes[:, 1])) ** 2
        k_values = torch.tensor([torch.sqrt(fx[n] * fy[n] * real_bbox[0] * real_bbox[1] / area[n]) for n in range(batch_size)]).to(torch.float32)
        k_values = cast(k_values, device)

        pred_uvd, pred_keypoints3d_int = model(reg_images, reg_images, k_value=k_values, K=other_K)
        error3d_int = torch.norm(pred_keypoints3d_int - gt_keypoints3d, dim=2)
        error3d_int = cast(error3d_int, device)
        loss_error3d_int = torch.mean(error3d_int)
        loss_error3d_int.backward()
        optimizer.step()

        # 更新损失值到 AverageValueMeter
        losses_3dkp.add(loss_error3d_int.item())

        # 每隔 100 个 batch 记录一次损失值到 TensorBoard，并重置 AverageValueMeter
        if (batch_idx + 1) % 100 == 0:
            
            tb_writer.add_scalar("Loss/3d_error", losses_3dkp.mean, epoch * len(iterator) + batch_idx)
            new_row = {
                "epoch": epoch,
                "step": batch_idx + 1,
                "loss_3dkp": losses_3dkp.mean,
            }
            df = df.append(new_row, ignore_index=True)
            losses_3dkp.reset()  # 重置 AverageValueMeter

    # 每个 epoch 结束时，记录该 epoch 的平均损失值到日志文件
    df.to_csv(log_csv, index=False)

    # 更新学习率调度器
    lr_scheduler.step()

tb_writer.close()

Epoch: 0


  5%|▍         | 99/2187 [00:41<11:31,  3.02it/s] /tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
  9%|▉         | 199/2187 [01:15<11:30,  2.88it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 14%|█▎        | 299/2187 [01:49<10:36,  2.97it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 18%|█▊        | 399/2187 [02:22<09:39,  3.08it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versi

Epoch: 1


  5%|▍         | 99/2187 [00:33<11:18,  3.08it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
  9%|▉         | 199/2187 [01:06<10:50,  3.05it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 14%|█▎        | 299/2187 [01:40<10:21,  3.04it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 18%|█▊        | 399/2187 [02:13<09:38,  3.09it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 2


...

  5%|▍         | 99/2187 [00:33<11:48,  2.95it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
  9%|▉         | 199/2187 [01:06<10:51,  3.05it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 14%|█▎        | 299/2187 [01:40<10:29,  3.00it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 18%|█▊        | 399/2187 [02:13<10:11,  2.93it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 19


  5%|▍         | 99/2187 [00:34<11:37,  2.99it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
  9%|▉         | 199/2187 [01:08<11:41,  2.83it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 14%|█▎        | 299/2187 [01:43<10:35,  2.97it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 18%|█▊        | 399/2187 [02:17<10:19,  2.89it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio

Epoch: 20


  5%|▍         | 99/2187 [00:38<14:09,  2.46it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
  9%|▉         | 199/2187 [01:13<11:40,  2.84it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 14%|█▎        | 299/2187 [01:50<11:54,  2.64it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(new_row, ignore_index=True)
 18%|█▊        | 399/2187 [02:28<10:40,  2.79it/s]/tmp/ipykernel_811337/2580833925.py:157: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versio